# JAX-CrossCat Test Runner

Run the full test suite on a GPU/TPU-accelerated runtime (Colab, Kaggle, etc.).

**Instructions:**
1. Upload this notebook to Colab or Kaggle
2. Select a GPU/TPU runtime
3. Run all cells — results are printed inline

## 1. Setup — Install from GitHub

In [ ]:
%pip install uv -q
!git clone https://github.com/sambhal-labs/jaxcross.git /content/jaxcross 2>/dev/null || (cd /content/jaxcross && git pull)
%cd /content/jaxcross

# Checkout the branch to test (change to 'main' for stable)
BRANCH = "perf/packed-kernel-optimization"  # @param {type:"string"}
!git checkout {BRANCH} && git pull origin {BRANCH}

!uv sync --extra dev --extra gpu -q
print(f"Branch: {BRANCH}")
print("Setup complete.")

## 2. Verify GPU/TPU is available

In [ ]:
import jax

print(f"JAX version: {jax.__version__}")
print(f"Devices: {jax.devices()}")
print(f"Backend: {jax.default_backend()}")

import crosscat

print(f"jax-crosscat version: {crosscat.__version__}")

## 3. Run fast tests (excludes slow integration tests)

In [ ]:
!uv run pytest -m "not slow" -v --tb=short 2>&1 | tail -80

## 4. Run full suite (including slow integration/recovery tests)

⚠️ This takes ~15-30 min on a T4 GPU. Skip if you only need fast tests.

In [ ]:
!uv run pytest -v --tb=short 2>&1 | tail -80

## 5. Lint check

In [ ]:
!uv run ruff check . && echo "✅ Lint passed" || echo "❌ Lint failed"
!uv run ruff format --check . && echo "✅ Format OK" || echo "❌ Format issues"

## 6. Quick smoke test — end-to-end inference

Verifies core functionality: initialize, packed sweep, queries.

In [ ]:
import time

import jax
import jax.numpy as jnp

from crosscat import (
    column_partition_ari,
    dependence_matrix,
    generate_crosscat_data,
    impute_and_confidence,
    initialize,
    log_joint,
    pack_state,
    packed_gibbs_sweep,
    predictive_anomalousness,
    predictive_sample,
    unpack_state,
)
from crosscat.types import ColumnType

# Generate synthetic data with known structure
key = jax.random.key(42)
result = generate_crosscat_data(
    key,
    n_rows=200,
    column_types=[
        ColumnType.CONTINUOUS,
        ColumnType.CONTINUOUS,
        ColumnType.CATEGORICAL,
        ColumnType.BINARY,
    ],
    n_views=2,
    n_clusters=3,
)
data = result["data"]
col_types = result["column_types"]
true_col_assigns = result["true_column_assignments"]

print(f"Data shape: {data.shape}")
print(f"Column types: {col_types}")
print(f"True column assignments: {true_col_assigns}")

In [ ]:
# Initialize and run packed inference
key, k1, k2 = jax.random.split(key, 3)
state_or_states = initialize(k1, data, col_types)
if isinstance(state_or_states, list):
    state = state_or_states[0]
else:
    state = state_or_states
packed = pack_state(state, max_views=8, max_clusters=16)

print("Running packed Gibbs sweep (first call triggers JIT compilation)...")
t0 = time.time()
packed = packed_gibbs_sweep(k2, packed, data, n_sweeps=50)
t1 = time.time()
print(f"50 sweeps completed in {t1 - t0:.1f}s (includes JIT compilation)")

# Second run — should be much faster
key, k3 = jax.random.split(key)
t0 = time.time()
packed = packed_gibbs_sweep(k3, packed, data, n_sweeps=50)
t1 = time.time()
print(f"50 more sweeps in {t1 - t0:.1f}s (compiled)")

# Unpack and check recovery
state = unpack_state(packed, col_types, data=data)
score = log_joint(state, data)
ari = column_partition_ari(state, true_col_assigns)
print(f"\nLog joint: {score:.2f}")
print(f"Column partition ARI: {ari:.3f}")
print(f"Discovered {state.n_views} views")

In [ ]:
# Query the posterior
key, k4, k5, k6 = jax.random.split(key, 4)

# Predictive sampling
samples = predictive_sample(k4, state, data, query_cols=[0], n_samples=500)
print(f"Predictive samples for col 0: mean={jnp.mean(samples):.2f}, std={jnp.std(samples):.2f}")

# Anomaly detection
anomaly = predictive_anomalousness(k5, state, data, query_row=0)
print(f"Anomaly score (row 0): {anomaly:.3f}")

# Imputation
value, confidence = impute_and_confidence(k6, state, data, query_col=0)
print(f"Imputed col 0: {value:.2f} (confidence: {confidence:.2f})")

# Dependence matrix (Z-matrix)
z = dependence_matrix([state])
print(f"\nDependence matrix (Z-matrix):\n{z}")

print("\n✅ All smoke tests passed!")

## 7. Performance benchmark — packed kernel timing

Measures per-sweep timing at different scales to verify optimization impact.
Compare against baseline: ~38s/sweep at 100x65, ~238s/sweep at 1000x257.

In [ ]:
import time
import jax
import jax.numpy as jnp
from crosscat.model import initialize
from crosscat.packed.state import pack_state, unpack_state
from crosscat.packed.kernels import packed_gibbs_sweep
from crosscat.types import ColumnType

# Enable XLA persistent cache for faster recompilation
from crosscat.packed.aot_cache import enable_xla_cache
enable_xla_cache()

configs = [
    ("50x11 (small)", 50, 11, [ColumnType.CONTINUOUS]*5 + [ColumnType.BINARY]*3 + [ColumnType.CATEGORICAL]*3),
    ("100x65 (medium)", 100, 65, [ColumnType.BINARY]*60 + [ColumnType.CONTINUOUS]*4 + [ColumnType.CATEGORICAL]*1),
]

n_sweeps = 5
for label, n_rows, n_cols, col_types in configs:
    key = jax.random.key(0)
    data = jax.random.normal(key, (n_rows, n_cols))
    # Make binary columns 0/1 and categorical cols integer
    for j, ct in enumerate(col_types):
        if ct == ColumnType.BINARY:
            data = data.at[:, j].set((data[:, j] > 0).astype(jnp.float32))
        elif ct == ColumnType.CATEGORICAL:
            data = data.at[:, j].set(jnp.abs(data[:, j] * 3).astype(jnp.int32).astype(jnp.float32) % 5)

    k1, k2 = jax.random.split(key)
    state = initialize(k1, data, col_types)
    packed = pack_state(state)

    # Warm-up (JIT)
    t0 = time.time()
    packed = packed_gibbs_sweep(k2, packed, data, n_sweeps=1)
    jit_time = time.time() - t0

    # Timed sweeps
    k3 = jax.random.key(99)
    t0 = time.time()
    packed = packed_gibbs_sweep(k3, packed, data, n_sweeps=n_sweeps)
    sweep_time = time.time() - t0

    print(f"{label}: JIT={jit_time:.1f}s, {n_sweeps} sweeps={sweep_time:.1f}s ({sweep_time/n_sweeps:.2f}s/sweep)")

print("\nDone! Compare s/sweep values against pre-optimization baselines.")